#### Chroma
Chroma is a AI-native open-source vector database focused on developer productivity and happiness. Chroma is licensed under Apache 2.0.

https://python.langchain.com/v0.2/docs/integrations/vectorstores/

In [2]:
## Building a sample vectordb
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [4]:
loader = TextLoader("speech.txt")
data = loader.load()
data

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness be

In [6]:
# Spiltting the data into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
splits = text_splitter.split_documents(data)
splits

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no'),
 Document(metadata={'source': 'speech.txt'}, page_content='no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied'),
 Document(metadata={'source': 'speech.txt'}, page_content='We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.'),
 Document(metadata={'source': 'speech.txt'}, page_content='Just because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our'),
 Document(metadata={'source': 'speech.txt'}, 

In [8]:
embeddings = OllamaEmbeddings(model="gemma2:2b")
vectordb = Chroma.from_documents(splits, embeddings)
vectordb

In [10]:
## Querying the vectordb

query="What does the speaker believe is the main reason the United States should enter the war?"
docs=vectordb.similarity_search(query)
docs[0].page_content

'We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.'

In [11]:
## Saving to the disk
vectordb = Chroma.from_documents(splits, embeddings, persist_directory="./chroma_db")

In [12]:
## Loading from the disk
new_vectordb = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)
docs=new_vectordb.similarity_search(query)
docs[0].page_content

'We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.'

In [15]:
## Similarity search with Score
docs = new_vectordb.similarity_search_with_score(query)
docs

[(Document(id='de7d7b93-4eff-4cb4-a80d-90e0cda9ff0e', metadata={'source': 'speech.txt'}, page_content='We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.'),
  7948.1123046875),
 (Document(id='81f11fad-3b8a-484f-bde0-db45bd3f8240', metadata={'source': 'speech.txt'}, page_content='I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.'),
  8614.58203125),
 (Document(id='c4f3cb93-a2b3-4bb8-8ea6-ddffdd35d5a8', metadata={'source': 'speech.txt'}, page_content='of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
  8796.3779296875),
 (Document(id='96e2c460-064a-4fab-af5b-d5be863c7031', metadata={'source': 'speech.txt'}, page_content='when America is privileged to spend her blood and her might for the principles that gave her birth and happine

In [16]:
## Retriever option
retriever = new_vectordb.as_retriever()
retriever.invoke(query)[0].page_content

'We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.'